## TODO

### Schema, profile, and configuration

- [ ] Define the input contract separately from generated fields: allow optional CSV overrides, but otherwise default Metadata Version to configurable `Aardvark`, generate Modified as the JSON-production timestamp in UTC, default Provider to `American Geographical Society Library – UWM Libraries`, and default Suppressed to boolean `false`.
- [ ] Decide where workflow defaults and their override precedence live (for example, a versioned configuration file plus CLI options); reject unknown configuration keys and record the effective configuration in conversion diagnostics.
- [ ] Reconcile the general-workshop `Alternative Title` column with the AGSL profile label `AGSL Call Number`; either standardize the template on the AGSL label or support an explicit, tested alias without silently dropping values.
- [ ] Compare every local profile obligation with current OGM documentation, document intentional AGSL overrides and their practical GeoBlacklight/Solr impact, and otherwise align the profile to the community standard.
- [ ] Resolve the `gbl_dateRange_drsim` contradiction between the profile/schema array type and OGM guidance showing one bracketed string; verify actual GeoBlacklight/Solr behavior and raise an OGM community issue if the published sources remain inconsistent.
- [ ] Document that `dct_references_s` must be a JSON-encoded string inside the outer JSON document.

### Parsing and normalization

- [x] Parse boolean fields explicitly instead of using `bool(value)`, which incorrectly treats strings such as `"false"` as `True`.
- [x] Only convert floats to integers when they represent whole numbers; avoid truncating legitimate decimal values.
- [x] Make `_im` array conversion tolerate whitespace and integer-like values such as `"1922.0"`.
- [x] Strip surrounding whitespace from every pipe-delimited value in Python, even when OpenRefine is used upstream, and test values such as `Index maps | Topographic Maps`.
- [ ] Skip wholly blank CSV rows and test that they never create `nan_BL_Aardvark.json`; keep the blank template header-only.
- [ ] Warn or fail on unknown input columns instead of silently ignoring them, including accidental columns such as `Column1`; decide explicitly whether `Github View` is removed or mapped to a supported/local reference URI.

### Identifiers and output files

- [x] Validate `ID` before constructing filenames so missing values do not produce `nan_BL_Aardvark.json`.
- [x] Resolve and validate canonical `Identifier` and generated `ID` before constructing filenames so missing or conflicting values cannot produce invalid output filenames.
- [ ] Replace the heuristic filename parsing with explicit ARK parsing to prevent filename collisions.
- [x] Preserve and test the critical ARK distinction: AGSL `id` uses the modified `ark:-77981-...` permalink form, while `dct_identifier_sm` uses normalized `ark:/77981/...`; never apply one normalization rule to both fields.
- [ ] Add optional NOID mint-and-bind support behind a testable service interface, preserving supplied IDs and making retries idempotent so the same source record cannot receive multiple ARKs.
- [ ] Define a parent/child identifier policy for sub-ARKs, including how canonical paths such as `ark:/77981/{parent}/{child}` map to collision-free GeoBlacklight `id` values, filenames, NOID bindings, and parent/child Aardvark relations.
- [ ] Detect duplicate output filenames before files are overwritten.
- [ ] Decide how reruns should handle stale JSON files left in the output directory.
- [x] Remove the unused `index` variable or restore a deliberate index-based fallback.

### Spatial metadata and enrichment

- [ ] Treat Geometry as required by the current community JSON Schema. Define how it can be derived from source extents or Bounding Box, how raw west/south/east/north coordinates become `ENVELOPE(W,E,N,S)`, and how records with no derivable geometry are reported for review.
- [ ] Evaluate reusable bounding-box and controlled-value logic in `../Clean-Validate/04_clean-validate.ipynb`, but separate its raw `west,south,east,north` representation from Aardvark ENVELOPE ordering and replace silent corrections/defaults with logged, testable policy.
- [ ] Explore a reproducible Theme classification step using existing projects or controlled-vocabulary mappings; preserve human overrides and flag uncertain classifications rather than inventing them silently.

### Validation and tests

- [ ] Add validation and useful error messages for missing CSV columns, malformed identifiers, invalid numeric arrays, and absent input files.
- [ ] Validate every generated record against `../../schema/geoblacklight-schema-aardvark.json` before final output and report all row/field errors together; update tests whenever the pinned community schema changes.
- [ ] Add representative conversion tests covering defaults and overrides, booleans, both ARK forms, Unicode, trimmed arrays, missing and minted IDs, references, geometry derivation, schema failures, blank rows, unknown columns, and decimal values.


## Step 1. Import Modules

In [1]:
from decimal import Decimal, InvalidOperation
from pathlib import Path
import json

import pandas as pd

## Step 2. Specify the file paths

This workflow uses two different descriptions of Aardvark metadata:

- The **field profile** (`aardvark.csv`) maps human-readable CSV headings to Aardvark field names and types. The converter reads this file to decide how to transform each value.
- The **JSON Schema** (`../../schema/geoblacklight-schema-aardvark.json`) validates the finished JSON record. We will connect that separate validation step later.

Calling `aardvark.csv` a profile here keeps those two responsibilities distinct.

In [2]:
csv_file_path = Path('../csv2JSON/OpenIndexMaps_Aardvark_workshop.csv')  # the input CSV
reference_uris_file_path = Path('../aardvark-profile/referenceURIs.csv')  # reference URI labels and values
profile_file_path = Path('../aardvark-profile/aardvark.csv')  # friendly labels mapped to Aardvark fields and types
output_dir = Path('json_output')  # generated JSON records

## Step 3. Define the input-loading functions

These functions define how the converter will load its three tabular inputs:

- the CSV records being converted;
- the reference URI mappings used to construct `dct_references_s`;
- the Aardvark field profile used to map friendly CSV labels to JSON fields.

In [3]:
def load_csv_records(csv_path):
    """Load the source metadata records from a CSV file."""
    return pd.read_csv(csv_path)


def load_reference_uris(reference_uris_path):
    """Load reference labels and return a label-to-URI mapping."""
    reference_uris_data = pd.read_csv(reference_uris_path)

    return dict(
        zip(
            reference_uris_data["LABEL"],
            reference_uris_data["URI"],
        )
    )


def load_field_profile(profile_path):
    """Load the CSV profile that maps labels to Aardvark fields."""
    return pd.read_csv(profile_path)

## Step 4. Define transformation helpers

These functions transform individual metadata values without reading files or writing output.

In [4]:
AGSL_NAAN = "77981"


def ark_to_agsl_id(identifier):
    """Convert a canonical AGSL ARK to the modified (url-safe) GeoBlacklight id form."""
    if not isinstance(identifier, str):
        raise TypeError("The canonical ARK must be a string.")

    canonical_ark = identifier.strip()
    expected_prefix = f"ark:/{AGSL_NAAN}/"

    if not canonical_ark.startswith(expected_prefix):
        raise ValueError(
            f"Expected an AGSL ARK beginning with {expected_prefix!r}; "
            f"received {canonical_ark!r}."
        )

    noid = canonical_ark.removeprefix(expected_prefix)

    if not noid:
        raise ValueError("The canonical ARK is missing its NOID name.")

    if "/" in noid:
        raise NotImplementedError(
            "Sub-ARK conversion is not implemented yet. "
            f"Received {canonical_ark!r}."
        )

    return f"ark:-{AGSL_NAAN}-{noid}"

In [5]:
def resolve_ark_fields(identifier, supplied_id=None):
    """Build and cross-check the two Aardvark identifier fields."""
    if pd.isna(identifier) or not str(identifier).strip():
        raise ValueError("Identifier must contain a canonical AGSL ARK.")

    canonical_identifier = str(identifier).strip()
    generated_id = ark_to_agsl_id(canonical_identifier)

    if pd.notna(supplied_id) and str(supplied_id).strip():
        supplied_id = str(supplied_id).strip()

        if supplied_id != generated_id:
            raise ValueError(
                "Identifier and ID do not represent the same ARK: "
                f"{canonical_identifier!r} generates {generated_id!r}, "
                f"but the CSV contains {supplied_id!r}."
            )

    return {
        "id": generated_id,
        "dct_identifier_sm": [canonical_identifier],
    }

In [6]:
def parse_integer(value):
    """Parse an integer or integer-like value without truncating decimals."""
    text = str(value).strip()

    try:
        number = Decimal(text)
    except InvalidOperation as error:
        raise ValueError(f"Expected an integer; received {value!r}.") from error

    if not number.is_finite() or number != number.to_integral_value():
        raise ValueError(f"Expected a whole number; received {value!r}.")

    return int(number)

In [7]:
def split_multivalues(value, field_name):
    """Split a pipe-delimited CSV value into an Aardvark array."""
    values = [
        item.strip()
        for item in str(value).split("|")
    ]

    if field_name.endswith("_im"):
        return [parse_integer(item) for item in values]

    return values

In [8]:
def stringify_scalar(value):
    """Convert a scalar CSV value to text without truncating decimals."""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value)

In [9]:
def parse_boolean(value):
    """Convert common CSV Boolean representations to a JSON Boolean."""
    if isinstance(value, bool):
        return value

    text = str(value).strip().casefold()

    if text == "true":
        return True

    if text == "false":
        return False

    try:
        number = Decimal(text)
    except InvalidOperation as error:
        raise ValueError(
            f"Expected true, false, 1, or 0; received {value!r}."
        ) from error

    if number == 1:
        return True

    if number == 0:
        return False

    raise ValueError(
        f"Expected true, false, 1, or 0; received {value!r}."
    )

In [10]:
def build_references(row, reference_uri_dict):
    """Build the Aardvark reference dictionary for one CSV record."""
    references = {}

    for reference_label, reference_uri in reference_uri_dict.items():
        if pd.notna(row.get(reference_label)):
            references[reference_uri] = row[reference_label]

    return references

### Check the transformation helpers

These examples verify ARK conversion and identifier resolution, demonstrate rejection of unsupported or conflicting identifiers, and check multivalued field conversion.

In [11]:
assert (
    ark_to_agsl_id("ark:/77981/gmgs1j97737")
    == "ark:-77981-gmgs1j97737"
)

assert (
    ark_to_agsl_id("  ark:/77981/gmgs1j97737  ")
    == "ark:-77981-gmgs1j97737"
)

try:
    ark_to_agsl_id("ark:/12345/example")
except ValueError:
    pass
else:
    raise AssertionError("An ARK with a different NAAN should be rejected.")


try:
    ark_to_agsl_id(None)
except TypeError:
    pass
else:
    raise AssertionError("A non-string identifier should be rejected.")


try:
    ark_to_agsl_id("ark:/77981/")
except ValueError:
    pass
else:
    raise AssertionError("An ARK without a NOID name should be rejected.")


try:
    ark_to_agsl_id("ark:/77981/gmgs1j97737/child")
except NotImplementedError:
    pass
else:
    raise AssertionError("Sub-ARKs should be recognized as unsupported for now.")

ark_fields = resolve_ark_fields(
    identifier="ark:/77981/gmgs1j97737",
    supplied_id="ark:-77981-gmgs1j97737",
)

assert ark_fields == {
    "id": "ark:-77981-gmgs1j97737",
    "dct_identifier_sm": ["ark:/77981/gmgs1j97737"],
}

assert resolve_ark_fields(
    identifier="ark:/77981/gmgs1j97737"
) == ark_fields

try:
    resolve_ark_fields(
        identifier="ark:/77981/gmgs1j97737",
        supplied_id="ark:-77981-different",
    )
except ValueError:
    pass
else:
    raise AssertionError("Conflicting Identifier and ID values should fail.")

assert split_multivalues(
    "Maps|Datasets",
    "gbl_resourceClass_sm",
) == ["Maps", "Datasets"]

assert split_multivalues(
    "1922|1924|1927",
    "gbl_indexYear_im",
) == [1922, 1924, 1927]

assert split_multivalues(
    "eng",
    "dct_language_sm",
) == ["eng"]

assert split_multivalues(
    "Index maps | Topographic Maps",
    "gbl_resourceType_sm",
) == ["Index maps", "Topographic Maps"]

assert parse_integer("1922") == 1922
assert parse_integer(" 1922 ") == 1922
assert parse_integer("1922.0") == 1922

assert split_multivalues(
    "1922 | 1924.0 | 1927",
    "gbl_indexYear_im",
) == [1922, 1924, 1927]

try:
    parse_integer("1922.5")
except ValueError:
    pass
else:
    raise AssertionError("A decimal value must not be truncated to an integer.")

assert stringify_scalar(1922.0) == "1922"
assert stringify_scalar(1922.5) == "1922.5"
assert stringify_scalar("1922") == "1922"
assert stringify_scalar("Public") == "Public"

assert parse_boolean(True) is True
assert parse_boolean(False) is False
assert parse_boolean("TRUE") is True
assert parse_boolean("false") is False
assert parse_boolean(1) is True
assert parse_boolean(0) is False
assert parse_boolean("1.0") is True
assert parse_boolean("0.0") is False

try:
    parse_boolean("maybe")
except ValueError:
    pass
else:
    raise AssertionError("An unrecognized Boolean value should fail.")

geojson_row = pd.Series({
    "Format": "GeoJSON",
    "Download": "https://example.org/example.geojson",
    "Index Map": "https://example.org/example.geojson",
})

geojson_references = build_references(
    geojson_row,
    {
        "Download": "http://schema.org/downloadUrl",
        "GeoJSON": "http://geojson.org/geojson-spec.html",
        "Index Map": "https://openindexmaps.org",
    },
)

assert geojson_references == {
    "http://schema.org/downloadUrl": "https://example.org/example.geojson",
    "https://openindexmaps.org": "https://example.org/example.geojson",
}

### Construct an Aardvark record

This function transforms one CSV row into an Aardvark record. It receives
the field profile and reference mappings explicitly, which keeps it
independent of notebook state and file-loading behavior.

In [12]:
def construct_json_data(
    row,
    ark_fields,
    profile_data,
    reference_uri_dict,
):
    """Construct one Aardvark record from a CSV row."""
    json_data = ark_fields.copy()

    for _, profile_row in profile_data.iterrows():
        label = profile_row["Label"]
        field_name = profile_row["Field Name"]
        field_type = profile_row["Field Type"]

        # These fields have already been resolved from the canonical ARK.
        if field_name in {"id", "dct_identifier_sm"}:
            continue

        if field_name == "dct_references_s":
            references = build_references(row, reference_uri_dict)

            if references:
                json_data[field_name] = json.dumps(references)

        elif pd.notna(row.get(label)):
            source_value = row.get(label)

            if field_type == "Array":
                json_data[field_name] = split_multivalues(
                    source_value,
                    field_name,
                )
            elif field_type == "Boolean or string":
                json_data[field_name] = parse_boolean(source_value)
            else:
                json_data[field_name] = stringify_scalar(source_value)

    return json_data

## Step 5. Convert the CSV records

In [13]:
def convert_csv_to_json(csv_file_path, reference_uris_file_path, profile_file_path, output_dir):
    output_dir = Path(output_dir)

    csv_data = load_csv_records(csv_file_path)
    reference_uri_dict = load_reference_uris(reference_uris_file_path)
    profile_data = load_field_profile(profile_file_path)
    
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)

    # Iterate over each row in the CSV and generate JSON files
    conversion_errors = []
    records_written = 0

    for csv_line_number, (_, row) in enumerate(
        csv_data.iterrows(),
        start=2,
    ):
        try:
            ark_fields = resolve_ark_fields(
                identifier=row.get("Identifier"),
                supplied_id=row.get("ID"),
            )
            json_data = construct_json_data(
                row,
                ark_fields,
                profile_data,
                reference_uri_dict,
            )
        except (TypeError, ValueError, NotImplementedError) as error:
            conversion_errors.append(
                {
                    "csv_line_number": csv_line_number,
                    "error_type": type(error).__name__,
                    "message": str(error),
                }
            )
            continue
        
        # Match the existing edu.uwm metadata-aardvark filename convention
        id_part = json_data["id"]
        filename_id = id_part.removeprefix('ark:-77981-')
        if filename_id == id_part:
            filename_id = id_part.split('/')[-1].split('-')[-1]
        file_name = f"{filename_id}_BL_Aardvark.json"
        file_path = output_dir / file_name
        
        # Write the JSON data to a file
        with file_path.open('w', encoding="utf-8") as json_file:
            json.dump(json_data, json_file, indent=4, ensure_ascii=False)

        records_written += 1

    return {
        "records_written": records_written,
        "records_skipped": len(conversion_errors),
        "errors": conversion_errors,
    }  

## Step 6: Run the script

In [14]:
conversion_result = convert_csv_to_json(
    csv_file_path,
    reference_uris_file_path,
    profile_file_path,
    output_dir,
)

print(f"Records written: {conversion_result['records_written']}")
print(f"Records skipped: {conversion_result['records_skipped']}")

for error in conversion_result["errors"]:
    print(
        f"CSV line {error['csv_line_number']} "
        f"({error['error_type']}): {error['message']}"
    )

print(f"JSON files generated in directory: {output_dir}")

Records written: 38
Records skipped: 0
JSON files generated in directory: json_output
